### 04. 레이어 합성·통합지수 (P3)

Layer 1(침수취약성) · Layer 3(노출·취약성·대응역량) · CDRI 를 만든 과정과 그 근거 수치를 본다.
계산은 `src/data/layers.py` 와 `src/stages/h06_layers.py` · `h07_cdri.py` 가 한다. 여기서는
**파이프라인이 쓴 것과 같은 함수·같은 산출물**을 불러 중간값을 확인한다. 노트북에서 새로 계산하지 않는다.

수식과 선택 근거는 `docs/METHODOLOGY.md` §3~§5.

In [1]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import layers as L
from src.pipeline.graph import Graph
from src.pipeline.report import node_metrics, node_table

GRAPH = Graph.load()
LAYERS = ROOT / "data/processed/layers"
pd.set_option("display.width", 200)

### P3 노드 상태

승인 대기(`awaiting_approval`)는 실패가 아니다. 사람이 확인해야 하류가 열린다.

In [2]:
node_table(*GRAPH.select(phase='P3'))

,이름,게이트,상태,검증,승인,실패 시,node_id
0,데이터 접근신청 현황,H00,pass,code / human,R1,h00_access_requests,h00_access_requests
1,강수량 수집 확인,H00,pass,code / human,-,h00_collect_rainfall,h00_collect_rainfall
2,하천수위 수집 확인,H00,pass,code / human,-,h00_collect_river,h00_collect_river
3,펌프장·하천 목록 수집 확인,H00,pass,code / human,-,h00_collect_small_tables,h00_collect_small_tables
4,SGIS 인구 통계·경계 수집 확인,H00,pass,code / human,-,h00_collect_sgis,h00_collect_sgis
5,토지피복·DEM 수집 확인,H00,pass,code / human,-,h00_collect_geo,h00_collect_geo
6,강수량 원본 검증,H01,pass,code / code / human,R1,h00_collect_rainfall,h01_contract_rainfall
7,하천수위 원본 검증,H01,pass,code / code / human,R1,h00_collect_river,h01_contract_river
8,펌프장·하천 목록 원본 검증,H01,pass,code / code / human,R1,h00_collect_small_tables,h01_contract_small_tables
9,SGIS 인구 통계·경계 원본 검증,H01,pass,code / code / human,R1,h00_collect_sgis,h01_contract_sgis


## 1. Layer 1 — 기후노출 × 도시민감도

국토부 「도시 기후변화 재해취약성분석 지침」 구조다. 두 축을 각각 z-score 합산한 뒤
Jenks 4등급으로 나누고, 두 등급의 합으로 취약성 I~IV 를 준다.

In [3]:
m1 = node_metrics("h06_layer1_flood")
print("격자", f"{m1['n_grid']:,}", "| 순위대상", f"{m1['n_universe']:,}")
pd.DataFrame(m1["class_counts"], index=["전체"]).T.rename(columns={"전체": "격자 수"})

격자 75,400 | 순위대상 10,202


,격자 수
I,4705
II,7897
III,33995
IV,28803


### 1-1. 강수 보간 — 지수 p 를 교차검증으로 고른 근거

IDW 의 거리 지수 $p$ 를 임의로 정하지 않고, 지점을 하나씩 빼고 나머지로 예측해
오차(RMSE)가 가장 작은 값을 골랐다. 변수마다 다른 $p$ 가 뽑힌다.

In [4]:
rows = []
for name, d in m1["idw"]["by_variable"].items():
    row = {"변수": name, "지점수": d["n_stations"], "선택 p": d["power"],
           "지점평균": d["station_mean"], "격자평균": d["grid_mean"],
           "반경밖 대체": d["n_nearest_fallback"]}
    row.update({f"RMSE p={k}": v for k, v in d["loocv_rmse"].items()})
    rows.append(row)
pd.DataFrame(rows)

,변수,지점수,선택 p,지점평균,격자평균,반경밖 대체,RMSE p=1.0,RMSE p=2.0,RMSE p=3.0
0,rain_annual_max_1h,29,1.0,40.803,40.330,514,4.0291,4.0695,4.2107
1,rain_hours_over_30mm,29,1.0,2.176,2.119,514,0.4740,0.4822,0.5012
2,rain_top5_3h,29,3.0,118.480,116.732,514,10.4254,9.4224,9.1525
3,rain_top5_24h,29,2.0,252.568,250.786,514,17.2311,16.7285,17.1753


### 1-2. 민감도 변수의 부호

부호는 계산 **전에** 물리적 근거로 고정했다. 결과를 보고 바꾸지 않는다.
`-1` 은 값이 작을수록 취약하다는 뜻이다(저지대·완경사).

In [5]:
from src.stages.h06_layers import EXPOSURE_SPEC, SENSITIVITY_SPEC

pd.DataFrame([
    {"축": "기후노출" if k in EXPOSURE_SPEC else "도시민감도", "변수": k, "부호": v,
     "z 평균": d["z_mean"], "z 표준편차": d["z_std"]}
    for group in ("exposure", "sensitivity")
    for k, d in m1["composite"][group].items()
    for v in [d["sign"]]
])

,축,변수,부호,z 평균,z 표준편차
0,기후노출,rain_annual_max_1h,1,0.0,1.0
1,기후노출,rain_hours_over_30mm,1,-0.0,1.0
2,기후노출,rain_top5_3h,1,-0.0,1.0
3,기후노출,rain_top5_24h,1,-0.0,1.0
4,도시민감도,rel_elev_m,-1,0.0,1.0
5,도시민감도,slope_deg,-1,0.0,1.0
6,도시민감도,twi,1,-0.0,1.0
7,도시민감도,impervious_frac,1,0.0,1.0
8,도시민감도,river_proximity,1,-0.0,1.0
9,도시민감도,culvert_proximity,1,-0.0,1.0


### 1-3. Jenks 등급 경계와 취약성 매트릭스

Jenks 는 급간 내부 편차제곱합을 최소화하는 경계를 찾는다. 구현이 맞는지는
작은 표본에서 완전탐색 최적해와 대조해 확인했다(`tests/test_layers.py`).

In [6]:
print("노출 등급 경계   :", m1["jenks_breaks"]["exposure"])
print("민감도 등급 경계 :", m1["jenks_breaks"]["sensitivity"])
print()
matrix = pd.DataFrame(
    [[L.ROMAN[L.VULNERABILITY_MATRIX[e + s]] for s in range(1, 5)] for e in range(1, 5)],
    index=[f"노출 {i}등급" for i in range(1, 5)],
    columns=[f"민감도 {j}등급" for j in range(1, 5)],
)
print("취약성 매트릭스 (I 이 가장 취약)")
matrix

노출 등급 경계   : [-2.1485, 0.8003, 3.7842, 8.6334]
민감도 등급 경계 : [-2.0589, 2.244, 7.9783, 26.979]

취약성 매트릭스 (I 이 가장 취약)


,민감도 1등급,민감도 2등급,민감도 3등급,민감도 4등급
노출 1등급,IV,IV,III,III
노출 2등급,IV,III,III,II
노출 3등급,III,III,II,I
노출 4등급,III,II,I,I


### 1-4. 강건성 — 하천 근접·침수예상도를 빼도 순위가 유지되는가

홍재주 외(2015)가 지적한 "하천 인접도에 따른 I등급 과다"와 예상도 의존을 확인한다.
두 변수를 빼고 다시 계산해도 순위 상관이 높으면 특정 변수에 끌려가지 않는다는 뜻이다.

In [7]:
es = m1["exclusion_sensitivity"]
print("제외 변수 :", es["excluded"])
print("순위 상관 :", es["spearman_rho_universe"])
print()
print("검증 상태")
print("  사례지 face-validity :", m1["case_study"])
print("  침수흔적 라벨        :", m1.get("label_available"), "|", m1.get("label_note", ""))

제외 변수 : ['river_proximity', 'culvert_proximity', 'flood_l210_100_depth_m']
순위 상관 : 0.9043

검증 상태
  사례지 face-validity : {'available': True, 'mapping': {'양덕동': ['양덕1동', '양덕2동'], '봉암동': ['봉암동'], '팔용동': ['팔룡동']}, 'unmatched_legal_dong': ['명서동', '사화동'], 'dong_found': ['봉암동', '양덕1동', '양덕2동', '팔룡동'], 'dong_missing': [], 'n_grid': 428, 'high_grade_share': 0.6098, 'base_rate': 0.3747, 'lift': 1.627, 'lift_min': 1.5, 'note': '선행연구 사례지는 독립 성능검증이 아니라 face-validity 점검이다 (하네스 §7)'}
  침수흔적 라벨        : True | 합산 AUC 는 기준을 넘었으나 2006, 2025 사상은 기준 미달이다. 성능을 서술할 때 사상별 표(trace.by_event)를 함께 싣는다


## 2. Layer 3 — 노출·취약성·대응역량

노출은 **수**, 취약성은 **비율**로 나눈다. 같은 사람을 두 번 세지 않기 위해서다.

In [8]:
m3 = node_metrics("h06_layer3_vuln")
print("연령 코드북 :", m3["elderly"]["age_codebook"])
print("검증 방법   :", m3["elderly"]["codebook_verified"])
print()
pd.Series({
    "집계구 수": m3["elderly"]["n_aggregation_units"],
    "집계구 조인율(순위대상)": m3["elderly"]["join_rate_universe"],
    "시 전체 65+ 비율": m3["elderly"]["city_elderly_share"],
    "격자 인구가중 65+ 비율": m3["elderly"]["grid_pop_weighted_universe"],
    "격자 단순평균 65+ 비율": m3["elderly"]["grid_unweighted_mean_universe"],
}).to_frame("값")

연령 코드북 : in_age 5세 계급, 65세 이상 = in_age_014 이후
검증 방법   : 노령화지수(to_in_004) 항등식 대조 — src/data/sgis.py 주석



,값
집계구 수,2118.0000
집계구 조인율(순위대상),1.0000
시 전체 65+ 비율,0.1841
격자 인구가중 65+ 비율,0.1908
격자 단순평균 65+ 비율,0.3173


**단순평균(0.32)이 시 전체(0.18)보다 훨씬 높은 이유.** 면적이 넓은 농촌 집계구가 격자를 많이
차지하기 때문이다. 시 전체와 비교할 수 있는 값은 **인구가중 평균**(0.19)이고, 그것이 맞는다.
집계구 하나가 최대 몇 개 격자에 같은 값을 주는지도 함께 기록한다 — 배분 불확실성이다.

In [9]:
print("집계구당 순위대상 격자 수 :", m3["elderly"]["grids_per_aggregation_unit"])
print()
print("대응역량")
cap = m3["capacity"]
pd.Series({
    "대피장소": cap["by_kind"]["shelter"], "방재기관": cap["by_kind"]["facility"],
    "대피소 중앙거리(m)": cap["shelter_dist_median_universe"],
    "방재기관 중앙거리(m)": cap["facility_dist_median_universe"],
    "부족도 평균": cap["capacity_deficit_mean_universe"],
}).to_frame("값")

집계구당 순위대상 격자 수 : {'median': 2.0, 'max': 106, 'note': '집계구 하나가 격자 여러 개에 같은 비율을 준다 — 배분 불확실성 (ANALYSIS_PLAN §4)'}

대응역량


,값
대피장소,444.0000
방재기관,516.0000
대피소 중앙거리(m),360.4000
방재기관 중앙거리(m),348.0000
부족도 평균,0.2361


**취약성 V 에서 빠진 변수.** 자료를 아직 못 구했다. 보고서 한계에 그대로 쓴다.

In [10]:
pd.Series(m3['missing_variables']).to_frame('사유')

,사유
one_person_household_ratio,SGIS 1인가구 미보유 (100m·집계구 모두)
old_building_ratio,GIS건물통합정보 SHP 미확보 (V-World 키 필요)
basement_building_count,건축물대장 지하층수 미확보 (건축HUB 키 필요)


## 3. CDRI 통합

$$\text{CDRI}_i = \prod_{k \in \{H,E,V,D\}} (X'_{ik})^{w_k}, \qquad
X' = 0.05 + 0.95\cdot\text{minmax}(X)$$

하한을 0 이 아니라 0.05 로 두는 이유는, minmax 로 0 이 된 요소 하나 때문에 기하평균 전체가
0 이 되어 순위 정보가 사라지는 것을 막기 위해서다.

In [11]:
m7 = node_metrics("h07_cdri")
print("기본 산식      :", m7["primary_formula"])
print("Layer 2 포함   :", m7["layer2_included_in_primary"], "|", m7["layer2_reason"])
print()
pd.DataFrame(m7["weights"], index=["H", "E", "V", "D"]).T

기본 산식      : geometric_equal
Layer 2 포함   : False | 관로 비공개로 검증 불가 — docs/decisions/001-layer2-design.md



,H,E,V,D
equal,0.2500,0.2500,0.2500,0.2500
entropy,0.0856,0.6385,0.1248,0.1511


**엔트로피 가중은 인구(E)에 64% 를 몰아준다.** 인구 분포의 왜도가 크기 때문이다.
이론 틀(네 요소가 모두 필요조건)과 어긋나므로 기본 산식으로 쓰지 않고 민감도 비교용으로만 쓴다.

### 3-1. 민감도 — 이 연구의 핵심 발견

In [12]:
pd.DataFrame(m7["variants"])

,variant,spearman_rho_vs_primary,top20_overlap,top20_overlap_pct
0,geometric_equal,1.0000,20,1.00
1,additive_equal,0.8781,10,0.50
2,geometric_entropy,0.8477,10,0.50
3,additive_entropy,0.9249,9,0.45


In [13]:
rb = m7["robustness"]
print("중위 순위상관 :", rb["median_spearman_rho"], f"(기준 {rb['min_spearman_required']})")
print("TOP20 최소중첩:", rb["min_top20_overlap"], f"(기준 {rb['min_overlap_required']})")
print("미달 항목     :", rb["unmet"])
print()
print("→ ranking_mode =", m7["ranking_mode"])
print("  ", m7["ranking_mode_note"])
print("  강건 공통집합:", m7["robust_core"])

중위 순위상관 : 0.8781 (기준 0.8)
TOP20 최소중첩: 9 (기준 14)
미달 항목     : ['TOP 20 최소 중첩 9 < 14']

→ ranking_mode = tier
   정밀 순위를 주장하지 않는다. 위험군(tier)과 강건 공통집합으로만 보고한다 — 하네스 H07 분기
  강건 공통집합: {'n': 8, 'note': '가중치·집계형 변형 4개 모두의 TOP 20 에 공통으로 드는 격자'}


전체 순위 경향은 잘 유지되는데($\rho \ge 0.85$) 최상위 20곳은 절반만 겹친다.
상위권은 값 차이가 미세해 산식을 조금만 바꿔도 순서가 뒤집히기 때문이다.

하네스 H07 은 이 경우 **"정밀 순위 대신 위험군 모드로 보고, 재튜닝 금지"** 라고 미리 정했다.
그래서 기준을 낮추지 않고 산출물의 성격을 바꿨다.

### 3-2. 그 밖의 민감도

In [14]:
print("해상도(MAUP) :", m7["maup"])
print()
g = m7["grade_system"]
print("등급 본안 :", g["scheme"], "—", g["scheme_reason"])
print("가중 kappa:", g["weighted_kappa"])
print()
pd.DataFrame({
    "Jenks(본안)": {k: v["n"] for k, v in g["raw"].items()},
    "규칙 적용 후": {k: v["n"] for k, v in g["final"].items()},
    "고정 백분위": {k: v["n"] for k, v in g["percentile_for_comparison"].items()},
    "Balica": {k: v["n"] for k, v in g["balica_for_comparison"].items()},
}).reindex(["R5", "R4", "R3", "R2", "R1"])

해상도(MAUP) : {'resolution_m': 500, 'n_blocks': 1605, 'spearman_rho_mean_vs_max': 0.8812, 'note': '500m 블록의 평균과 최대 재집계 순위 비교'}

등급 본안 : calibration — 침수흔적 양성 격자 299개 ≥ 100 이고 시간 분할 가능
가중 kappa: {'jenks_vs_balica': 0.7199, 'jenks_vs_percentile': 0.8104, 'balica_vs_percentile': 0.4875, 'note': '2차 가중 kappa. Landis & Koch(1977) 기준 0.61~0.80 substantial', 'calibration_vs_jenks': 0.8018, 'calibration_vs_balica': 0.4738}



,Jenks(본안),규칙 적용 후,고정 백분위,Balica
R5,70,124,205,226
R4,950,953,816,1746
R3,2041,2048,2040,5831
R2,3060,3018,3060,2398
R1,4081,4059,4081,1


### 3-3. 주 원인을 백분위로 정한 이유

가법형 구성비의 최댓값을 쓰면 분포가 치우친 요소(인구)가 거의 항상 이겨서 조치가 한쪽으로 쏠린다.
그래서 ANALYSIS_PLAN §5 대로 **구성요소 백분위가 가장 높은 것**을 주 원인으로 쓴다.

In [15]:
print(m7["cdri_summary"]["primary_cause_rule"])
print()
pd.Series(m7["cdri_summary"]["primary_cause_counts"]).to_frame("격자 수")

구성요소 백분위가 가장 높은 것 (ANALYSIS_PLAN §5)



,격자 수
V,2945
E,2778
H,2278
D,2201


### 3-4. 구성요소 상관 — 중복 투입이 없는지 확인

In [16]:
cdri = gpd.read_file(LAYERS / "cdri.gpkg", layer="cdri")
cdri[["H_scaled", "E_scaled", "V_scaled", "D_scaled", "cdri"]].corr().round(3)

,H_scaled,E_scaled,V_scaled,D_scaled,cdri
H_scaled,1.000,0.348,-0.401,-0.095,0.548
E_scaled,0.348,1.000,-0.519,-0.178,0.729
V_scaled,-0.401,-0.519,1.000,0.092,-0.142
D_scaled,-0.095,-0.178,0.092,1.000,0.134
cdri,0.548,0.729,-0.142,0.134,1.000


## 4. 침수흔적 외적 검증 — 실제로 잠겼던 곳과 대조

여기부터가 "지도를 만들었다"와 "실제로 맞췄다"를 가르는 부분이다.
흔적도는 Layer 1 의 **입력이 아니다**. 산출이 끝난 뒤 채점에만 쓴다 (ANALYSIS_PLAN §2-3).


In [17]:
trace = m1["trace"]

print("자료원별 건수 :", trace["rows_per_file"])
print("중복 도형 제거:", trace["duplicate_geometries_dropped"], "건 (L100 침수심 = L110 침수위)")
print("최종 흔적     :", trace["n_traces"], "건 /", trace["area_km2"], "km2")
print("사상          :", trace["n_events"], "건 /", trace["n_years"], "개 연도", trace["years"])
print("일자 범위     :", trace["date_range"])
print()
print("양성 격자     :", trace["n_labelled_grid"], "(순위대상", trace["n_labelled_in_universe"], ")")
print("판정 기준     :", trace["min_positive_required"], "칸 →",
      "충족" if trace["label_sufficient"] else "미달")

자료원별 건수 : {'L100_침수심.shp': 19, 'L110_침수위.shp': 18, 'changwon_flood_traces.gpkg': 125}
중복 도형 제거: 17 건 (L100 침수심 = L110 침수위)
최종 흔적     : 145 건 / 4.619 km2
사상          : 12 건 / 6 개 연도 ['2006', '2012', '2014', '2016', '2019', '2025']
일자 범위     : ['2006-07-10', '2025-07-19']

양성 격자     : 642 (순위대상 299 )
판정 기준     : 30 칸 → 충족


### 4-1. 두 기준 모두 통과

기준은 자료를 보기 **전에** `config/config.yaml` 에 박아 둔 값이다. 사후에 낮추지 않는다.


In [18]:
top_all, top_uni = trace["top20pct_all_grid"], trace["top20pct_universe"]
pd.DataFrame({
    "전체 격자": [trace["auc_all_grid"], top_all["capture_rate"], top_all["lift"], top_all["base_rate"]],
    "순위대상": [trace["auc_universe"], top_uni["capture_rate"], top_uni["lift"], top_uni["base_rate"]],
    "기준": [trace["auc_min"], trace["capture_min"], "—", "—"],
}, index=["ROC-AUC", "상위 20% 포착률", "lift", "기저 발생률"])

,전체 격자,순위대상,기준
ROC-AUC,0.7504,0.7134,0.7
상위 20% 포착률,0.5436,0.4415,0.5
lift,2.7180,2.2080,—
기저 발생률,0.0085,0.0293,—


**읽는 법.** 상위 20% 격자가 실제 침수의 54.4%를 담았다. 무작위로 20%를 고르면 20%만
담긴다. 즉 2.72배 효율이다.

순위대상만 보면 낮아진다(AUC 0.713). 인구·주택이 있는 칸끼리는 서로 비슷해 변별이 어렵기
때문이다. **두 값을 모두 보고한다** — 전체 기준만 쓰면 "산지를 빼서 얻은 점수"라는 지적을
받고, 순위대상만 쓰면 지수가 저지대를 골라내는 능력을 과소평가한다.


### 4-2. 9/16 의 AUC 0.365 는 무엇이었나

창원시 회신분만 넣었을 때는 0.365 — 무작위보다 낮았다. 지수를 고쳐서 0.750 이 된 것이
**아니다.** 가중치·산식은 그대로고 표본이 농촌에서 시가지로 바뀌었다.


In [19]:
diag = trace["diagnosis"]
pd.DataFrame({
    "침수 격자 중앙값": diag["flooded_median"],
    "창원 전체 중앙값": diag["city_median"],
}).rename_axis("변수")

,침수 격자 중앙값,창원 전체 중앙값
변수,,
elev_m,6.310,83.152
slope_deg,0.497,11.351
twi,10.928,7.163
impervious_frac,0.450,0.000
flood_l210_100_frac,0.080,0.000
pump_dist_m,3608.608,5571.945
pop_total,0.000,0.000


지형 세 변수(표고·경사·TWI)는 두 표본 모두에서 저지대·평지·물 모임을 정확히 잡았다.
갈린 것은 **도시 변수**다. 9/16 표본은 불투수면 0%, 예상 침수심 0, 펌프장 14.7km 밖의
농경지였다. Layer 1 은 도시 내수침수 지수이므로 거기서는 점수가 낮을 수밖에 없다.

따라서 0.365 는 반증이 아니라 **적용 범위의 경계**였다. 이 지수는 시가지용이다.


In [20]:
print("시 침수예상도가 덮은 흔적 격자:", diag["n_covered_by_city_flood_map"], "/", trace["n_labelled_grid"])
print()
print("시간 분할 (등급 캘리브레이션과 검증의 순환을 막는 장치)")
for k, v in trace["time_split"].items():
    print(f"  {k:20} {v}")

시 침수예상도가 덮은 흔적 격자: 334 / 642

시간 분할 (등급 캘리브레이션과 검증의 순환을 막는 장치)
  years                ['2006', '2012', '2014', '2016', '2019', '2025']
  calibration_years    ['2006', '2012', '2014']
  validation_years     ['2016', '2019', '2025']
  n_cal                440
  n_val                251
  usable               True
  n_both               49


### 4-3. 사상별로 나눠 보면 성능이 갈린다

합산 AUC 는 큰 사상 하나가 좋으면 나머지가 나빠도 높게 나온다. 사상 하나씩 따로 채점해
성능이 특정 호우에 기대고 있는지 본다. 음성은 **어느 사상에서도 잠기지 않은 격자**로 둔다 —
다른 사상에서 잠긴 칸을 음성으로 세면 맞힌 것을 틀렸다고 채점하게 된다.


In [21]:
rows = []
for r in trace["by_event"]:
    m = r["median"]
    rows.append({
        "사상": r["event_name"],
        "양성 격자": r["n_positive_grid"],
        "AUC": r.get("auc"),
        "상위20% 포착률": r["top20pct"]["capture_rate"] if "auc" in r else None,
        "불투수면": m["impervious_frac"],
        "예상 침수심": m["flood_l210_100_frac"],
        "펌프장 km": round(m["pump_dist_m"] / 1000, 1),
        "내수 비율": r["inland_share"],
    })
pd.DataFrame(rows, index=[r["event_year"] for r in trace["by_event"]]).rename_axis("연도")

,사상,양성 격자,AUC,상위20% 포착률,불투수면,예상 침수심,펌프장 km,내수 비율
연도,,,,,,,,
2006,2006.7.3 제3호 태풍 에위니아로 인한 집,107,0.3894,0.0000,0.000,0.000,14.3,0.000
2012,태풍 산바,206,0.7924,0.5583,0.565,0.185,1.9,0.516
2014,2014년 국지성 집중호우,132,0.7864,0.5076,0.000,0.000,10.1,0.404
2016,"제8호 태풍 ""차바""",170,0.9204,0.9353,0.990,0.900,1.7,0.000
2019,미탁,56,0.9617,0.9821,1.000,0.820,0.3,0.000
2025,2025년 07.19 집중호우,25,0.3671,0.0000,0.000,0.000,14.7,0.900


**네 사상은 기준(0.70)을 넘고 두 사상은 무작위(0.5)보다 낮다.** 그 두 사상만 공통점이
있다 — 잠긴 곳이 불투수면 0%, 시 예상 침수심 0, 펌프장 14km 밖의 **농경지**다.

2014년은 불투수면 0%, 펌프장 10km 밖인데도 AUC 0.786 이다. 원인이 외수범람·하천범람이고
잠긴 격자의 하천 거리 중앙값이 100m(최솟값)였다. 도시 변수가 0이어도 **하천 근접 변수가
잡아낸** 경우다.

| 잡는다 | 못 잡는다 |
|---|---|
| 시가지 내수침수 (불투수면·펌프 서비스권) | 농경지 배수 불량 (펌프 서비스권 밖) |
| 하천 인접 범람 (하천 거리 변수) | — |

이것은 실패가 아니라 **적용 범위**다. Layer 1 은 설계 단계부터 국토부 지침의 **도시**
기후변화 재해취약성분석을 따랐고, 창원시 침수예상도도 시가지를 대상으로 만들어졌다.


In [22]:
spread = trace["event_auc_spread"]
for k, v in spread.items():
    print(f"{k:20} {v}")

n_scored             6
min                  0.3671
max                  0.9617
all_above_min        False
events_below_min     ['2006', '2025']
note                 합산 AUC 는 기준을 넘었으나 사상별로는 갈린다. 미달 사상을 함께 보고하지 않으면 성능을 과대 진술하게 된다


**합산값만 쓰면 과대 진술이 된다.** 코드가 이것을 강제한다 — 기준 미달 사상이 있으면
Layer 1 이 안내문에 그 목록을 박아 넣는다 (`_event_spread`).

사후에 범위를 좁혀 점수를 올리지 않았다. "시가지만 골라 다시 계산하면 AUC 가 오른다"는
계산은 하지 않는다 — 검증 자료를 본 뒤 대상을 고르는 것이기 때문이다.


## 5. 등급 경계 캘리브레이션 — 점수가 아니라 발생률로

Jenks 는 **점수 분포**의 단절만 본다. "R5 와 R4 가 실제로 얼마나 다르게 잠기는가"는
답하지 못한다. 결정 003 이 본안으로 고른 3안이 그 질문에 답한다.

순환을 막는 장치: 앞 사상(2006·2012·2014)으로 경계를 정하고 뒤 사상(2016·2019·2025)으로
채점한다. 분할 규칙은 결과를 보기 전에 정했다.


In [23]:
gs = m7["grade_system"]
print("본안 :", gs["scheme"])
print("사유 :", gs["scheme_reason"])
print()
cal = gs["calibration"]
print("경계 (R1~R4 상한):", [round(b, 4) for b in cal["breaks"]])
print("등급 크기        :", cal["grade_sizes"])
print("PAV 단절점 수    :", cal["n_pav_breaks"])
print()
pd.DataFrame(cal["moves"])

본안 : calibration
사유 : 침수흔적 양성 격자 299개 ≥ 100 이고 시간 분할 가능

경계 (R1~R4 상한): [0.3052, 0.4327, 0.5864, 0.8412]
등급 크기        : [4081, 3060, 2041, 950, 70]
PAV 단절점 수    : 7



,target_percentile,moved_to_percentile,shift_pp,snapped_to_pav_break
0,0.98,0.9931,1.31,True
1,0.90,0.9000,0.00,False
2,0.70,0.7000,0.00,False
3,0.40,0.4000,0.00,False


목표 백분위(98/90/70/40)에서 ±3%p 안에 PAV 곡선의 단차가 있으면 그리로 옮긴다.
R5 경계만 옮겨졌고(98 → 99.31 백분위) 나머지 셋은 단차가 없어 목표 백분위에 남았다.

### 5-1. 검증 사상 채점 (out-of-sample)


In [24]:
val = gs["validation"]["calibration"]
tbl = pd.DataFrame(val["rows"]).set_index("grade")
tbl["인접 비"] = [None] + val["adjacent_ratios"]
tbl.rename(columns={"n": "격자", "positives": "검증 양성",
                    "incidence": "발생률", "lift": "lift"})

,격자,검증 양성,발생률,lift,인접 비
grade,,,,,
1,4081,47,0.01152,0.653,NaN
2,3060,53,0.01732,0.982,1.503
3,2041,47,0.02303,1.305,1.330
4,950,24,0.02526,1.432,1.097
5,70,9,0.12857,7.287,5.090


In [25]:
print("기저 발생률   :", val["base_rate"])
print("단조 증가     :", val["monotone"])
print("추세검정      : z =", val["z"], ", p =", f'{val["p_value"]:.2e}')
print("최상/최하 비  :", val["top_over_bottom"], "배")
print()
print("판정 (기준 완화 없음)")
for k, ok in val["criteria"].items():
    print(f"  {k:26} {'통과' if ok else '미달'}")

기저 발생률   : 0.01764
단조 증가     : True
추세검정      : z = 5.4546 , p = 4.91e-08
최상/최하 비  : 11.16 배

판정 (기준 완화 없음)
  monotone                   통과
  trend_significant          통과
  all_adjacent_ratios_met    미달
  lift_top_met               통과
  lift_second_met            미달


### 5-2. 미달 기준은 지우지 않고 주장 범위를 줄인다

두 기준이 미달이다. 기준을 사후에 낮추는 것은 금지돼 있으므로(재튜닝 금지),
대신 **무엇을 주장해도 되는지**를 코드가 산출한다.


In [26]:
rc = gs["validation"]["reporting_constraint"]
for k, v in rc.items():
    print(f"{k:28} {v}")

monotone_claim_allowed       True
separable_grades             ['R5']
pairs_not_separable          ['R3-R4']
tier_structure_for_report    {'1': ['R1', 'R2', 'R3'], '2': ['R4'], '3': ['R5']}
failed_criteria              ['all_adjacent_ratios_met', 'lift_second_met']
note                         단조 추세는 검증 사상에서 확인됐다. 인접 발생률 비가 기준에 못 미친 등급쌍은 개별 등급으로 주장하지 않고 통합 단계로 보고한다 (CDRI_GRADE_SYSTEM §1③ ④).


- 말할 수 있는 것: **등급이 오를수록 실제로 더 자주 잠긴다** (검증 사상에서 확인)
- 말할 수 있는 것: **R5 는 기저의 7.3배, R1 의 11.2배로 잠긴다**
- 말할 수 **없는** 것: R3 과 R4 는 서로 다르다 — 발생률이 갈리지 않는다

자료가 뒷받침하는 실무 구조는 5단이 아니라 **3단**이다.


In [27]:
mg = gs["validation"]["calibration_merged"]
print("통합 구성:", mg["tier_members"])
pd.DataFrame(mg["rows"]).set_index("grade").rename(
    columns={"n": "격자", "positives": "검증 양성", "incidence": "발생률"})

통합 구성: {'1': ['R1', 'R2', 'R3'], '2': ['R4'], '3': ['R5']}


,격자,검증 양성,발생률,lift
grade,,,,
1,9182,147,0.01601,0.907
2,950,24,0.02526,1.432
3,70,9,0.12857,7.287


### 5-3. 경계가 파라미터에 휘둘리는가 (민감도)

±3%p 와 1.3 은 설계 파라미터다. 흔들어 보고 경계가 크게 달라지면 이 방식을 본안으로
쓰면 안 된다.


In [28]:
sens = pd.DataFrame(cal["sensitivity"])
sens["grade_sizes"] = sens["grade_sizes"].astype(str)
sens.drop(columns=["breaks"])

,tolerance,min_ratio,grade_sizes,n_merge_recommended
0,0.02,1.2,"[4081, 3060, 2041, 950, 70]",2
1,0.02,1.3,"[4081, 3060, 2041, 950, 70]",2
2,0.02,1.5,"[4081, 3060, 2041, 950, 70]",2
3,0.03,1.2,"[4081, 3060, 2041, 950, 70]",2
4,0.03,1.3,"[4081, 3060, 2041, 950, 70]",2
5,0.03,1.5,"[4081, 3060, 2041, 950, 70]",2
6,0.05,1.2,"[4081, 3060, 2527, 464, 70]",2
7,0.05,1.3,"[4081, 3060, 2527, 464, 70]",2
8,0.05,1.5,"[4081, 3060, 2527, 464, 70]",2


±2%p 와 ±3%p 에서 등급 크기가 **완전히 동일**하고, ±5%p 에서만 R3/R4 경계가 움직인다.
R5 경계는 아홉 조합 전부에서 70칸으로 고정이다. 경계는 안정적이다.

### 5-4. 검증에 진 방식으로 갈아타지 않은 이유


In [29]:
cmp = pd.DataFrame({
    name: {
        "단조성": v["monotone"],
        "최상위 lift": v["rows"][-1]["lift"],
        "최상/최하 비": v["top_over_bottom"],
        "미달 기준 수": len(v["failed_criteria"]),
    }
    for name, v in gs["validation"].items() if isinstance(v, dict) and "rows" in v
}).T
print(cmp.to_string())
print()
print("방식 간 일치도 (2차 가중 kappa)")
for k, v in gs["weighted_kappa"].items():
    if k != "note":
        print(f"  {k:28} {v}")

                      단조성 최상위 lift 최상/최하 비 미달 기준 수
calibration          True    7.287   11.16       2
calibration_merged   True    7.287    8.03       1
jenks               False    2.725    5.07       4

방식 간 일치도 (2차 가중 kappa)
  jenks_vs_balica              0.7199
  jenks_vs_percentile          0.8104
  balica_vs_percentile         0.4875
  calibration_vs_jenks         0.8018
  calibration_vs_balica        0.4738


캘리브레이션이 네 기준 모두에서 Jenks 보다 낫다. 그렇다고 **검증 결과를 보고 본안을
고르지는 않는다** — 그러면 검증 자료가 선택에 개입해 더는 out-of-sample 이 아니게 된다.
본안은 결정 003 이 자료를 보기 전에 정했고, 검증은 주장 범위만 정한다.
Jenks 수치는 비교 정보로만 싣는다.


## 6. 다음 단계
